# E17 文本 EDA：浅特征、词袋与分类基线

> 覆盖矩阵：文本分析 · API：`AdvancedFeatureEngineer.extract_text_features`

## 你将学到
1. 长度 / 词数等 **basic** 浅特征的业务含义。
2. `method='tfidf'` 进表格模型前的稀疏性与泄漏（矢量器只 fit 训练折）。
3. 中文分词仍是缺口（空格分词不适用于中文）。


In [ ]:
QUICK = True
RANDOM_STATE = 42
from wanglaoshi import ready
ready(seed=RANDOM_STATE)


## 1. 合成评论数据

In [ ]:
import numpy as np
import pandas as pd
from wanglaoshi import DataAnalyzer

rng = np.random.default_rng(RANDOM_STATE)
pos = [
    'great product fast shipping love it',
    'excellent quality highly recommend',
    'amazing value and wonderful support',
]
neg = [
    'terrible quality broke quickly',
    'awful experience never again',
    'poor packaging damaged item refund',
]
rows = []
for _ in range(80):
    rows.append({'text': rng.choice(pos) + ' ' + rng.choice(['', 'thanks', '!!!']), 'label': 1})
for _ in range(80):
    rows.append({'text': rng.choice(neg) + ' ' + rng.choice(['', 'disappointed', '...']), 'label': 0})
df = pd.DataFrame(rows).sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)
print(df.label.value_counts())
print(DataAnalyzer(df).get_basic_info().get('dtypes'))


## 2. extract_text_features(basic)

In [ ]:
from wanglaoshi import FeatureEngineering as FE
eng = FE.AdvancedFeatureEngineer()
feat = eng.extract_text_features(df, 'text', method='basic')
print([c for c in feat.columns if c.startswith('text_')])
print(feat[['text_length', 'text_word_count', 'label']].groupby('label').mean())


## 3. 折内 TF-IDF + 基线

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from pathlib import Path

Xtr, Xte, ytr, yte = train_test_split(df['text'], df['label'], test_size=0.3, random_state=RANDOM_STATE, stratify=df['label'])
vec = TfidfVectorizer(max_features=100, ngram_range=(1, 2))
Xtr_v = vec.fit_transform(Xtr)  # 只 fit 训练折
Xte_v = vec.transform(Xte)
clf = LogisticRegression(max_iter=500).fit(Xtr_v, ytr)
print(classification_report(yte, clf.predict(Xte_v), digits=3))
out = Path('outputs/17_text_eda'); out.mkdir(parents=True, exist_ok=True)
pd.DataFrame({'n_features': [Xtr_v.shape[1]], 'acc': [clf.score(Xte_v, yte)]}).to_csv(out / 'text_metrics.csv', index=False)


## 附录
- 作业：对中文语料说明为何 `str.split` 失败；调研 jieba 作为可选依赖。
- 包优化：文档强调矢量器折内 fit；可选 `method='tfidf'` 返回稀疏矩阵接口统一。
